In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root
PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

from environment.sae_warning_env import SAEWarningEnv

print("Project root:", PROJECT_ROOT)
print("Imports successful")

Project root: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning
Imports successful


In [2]:
train_df = pd.read_csv(
    PROJECT_ROOT / "data/processed/mimic3/sae_rl_train.csv"
)

test_df = pd.read_csv(
    PROJECT_ROOT / "data/processed/mimic3/sae_rl_test.csv"
)

print("TRAIN:", train_df.shape)
print("TEST :", test_df.shape)

print("\nTrain patients:", train_df["subject_id"].nunique())
print("Test patients :", test_df["subject_id"].nunique())

print("\nTrain future SAE:", train_df["future_sae_1h"].sum())
print("Test future SAE :", test_df["future_sae_1h"].sum())

TRAIN: (3302, 13)
TEST : (1254, 13)

Train patients: 20
Test patients : 5

Train future SAE: 22
Test future SAE : 13


In [3]:
train_env = SAEWarningEnv(train_df)
test_env = SAEWarningEnv(test_df)

print("TRAIN ENVIRONMENT")
print("Observation:", train_env.observation_space)
print("Actions:", train_env.action_space)

print("\nTEST ENVIRONMENT")
print("Observation:", test_env.observation_space)
print("Actions:", test_env.action_space)

TRAIN ENVIRONMENT
Observation: Box(-inf, inf, (7,), float32)
Actions: Discrete(3)

TEST ENVIRONMENT
Observation: Box(-inf, inf, (7,), float32)
Actions: Discrete(3)


In [4]:
state, info = train_env.reset(seed=42)

print("Initial state:")
print(state)

print("\nInfo:")
print(info)

next_state, reward, terminated, truncated, info = train_env.step(0)

print("\nAfter action 0:")
print("Next state:", next_state)
print("Reward:", reward)
print("Terminated:", terminated)
print("Info:", info)

Initial state:
[ 15.   10.  101.5  69.   21.   98.    1. ]

Info:
{'icustay_id': 206504, 'hour': 1}

After action 0:
Next state: [15. 15. 96. 69. 20. 98.  2.]
Reward: 1.0
Terminated: False
Info: {'icustay_id': 206504, 'hour': 1, 'future_sae': 0, 'action': 0}


In [5]:
# ============================================================
# SIMPLE CLINICAL BASELINE
# ============================================================

def gcs_baseline_action(row):

    gcs = row["gcs_last_observed"]

    if gcs <= 8:
        return 2       # escalation

    return 0           # routine monitoring

In [6]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = []
y_pred = []

for _, row in test_df.iterrows():

    action = gcs_baseline_action(row)

    # Warning actions 1 and 2 are treated as "warning"
    prediction = int(action > 0)

    y_true.append(
        int(row["future_sae_1h"])
    )

    y_pred.append(
        prediction
    )

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

print("GCS BASELINE")
print("--------------------")
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

GCS BASELINE
--------------------
Precision: 0.00938337801608579
Recall   : 0.5384615384615384
F1       : 0.01844532279314888

Confusion matrix:
[[502 739]
 [  6   7]]


In [7]:
import stable_baselines3

print(
    "Stable-Baselines3 version:",
    stable_baselines3.__version__
)

Stable-Baselines3 version: 2.4.1


Stable-Baselines3: 2.4.1


NameError: name 'model' is not defined

In [11]:
from stable_baselines3 import DQN

# Create training environment
train_env = SAEWarningEnv(train_df)

model = DQN(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=1e-3,
    buffer_size=5000,
    learning_starts=100,
    batch_size=32,
    gamma=0.95,
    train_freq=4,
    target_update_interval=500,
    exploration_fraction=0.3,
    exploration_final_eps=0.05,
    verbose=1,
    seed=42
)

print("DQN model created")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
DQN model created


In [12]:
import stable_baselines3 as sb3

print("Stable-Baselines3:", sb3.__version__)

state, info = test_env.reset(seed=123)
state = np.asarray(state, dtype=np.float32)

action, _ = model.predict(
    state,
    deterministic=True
)

print("DQN prediction successful!")
print("Action:", int(action))

Stable-Baselines3: 2.4.1
DQN prediction successful!
Action: 2


In [13]:
from stable_baselines3 import DQN

# Create training environment
train_env = SAEWarningEnv(train_df)

model = DQN(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=1e-3,
    buffer_size=5000,
    learning_starts=100,
    batch_size=32,
    gamma=0.95,
    train_freq=4,
    target_update_interval=500,
    exploration_fraction=0.3,
    exploration_final_eps=0.05,
    verbose=1,
    seed=42
)

print("DQN model created")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
DQN model created


In [14]:
model_path = (
    PROJECT_ROOT /
    "models" /
    "dqn_sae_warning"
)

model_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

model.save(
    str(model_path)
)

print("Model saved to:")
print(model_path)

Model saved to:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/models/dqn_sae_warning


In [15]:
import numpy as np
import torch
import stable_baselines3 as sb3

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Stable-Baselines3:", sb3.__version__)

x = np.array([1, 2, 3], dtype=np.float32)

print("\nNumPy test:")
print(type(x), x.dtype)

print("\nPyTorch conversion test:")
try:
    y = torch.as_tensor(x)
    print("SUCCESS:", y)
    print("dtype:", y.dtype)
except Exception as e:
    print("FAILED:", repr(e))

NumPy: 1.26.4
PyTorch: 2.2.2
Stable-Baselines3: 2.4.1

NumPy test:
<class 'numpy.ndarray'> float32

PyTorch conversion test:
SUCCESS: tensor([1., 2., 3.])
dtype: torch.float32


In [16]:
import numpy as np
import torch

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

x = np.array([1, 2, 3], dtype=np.float32)

try:
    y = torch.as_tensor(x)

    print("\nPyTorch ↔ NumPy test: SUCCESS")
    print("Tensor:", y)
    print("Tensor dtype:", y.dtype)

except Exception as e:
    print("\nFAILED:")
    print(repr(e))

NumPy: 1.26.4
PyTorch: 2.2.2

PyTorch ↔ NumPy test: SUCCESS
Tensor: tensor([1., 2., 3.])
Tensor dtype: torch.float32


In [17]:
# ============================================================
# FIX SB3 / NUMPY OBSERVATION DTYPE
# ============================================================

state, info = test_env.reset(seed=123)

# Force observation into a standard float32 NumPy array
state = np.asarray(state, dtype=np.float32)

print("State:", state)
print("State type:", type(state))
print("State dtype:", state.dtype)
print("State shape:", state.shape)

# Verify SB3 can accept the observation
action, _ = model.predict(
    state,
    deterministic=True
)

print("\nDQN prediction successful!")
print("Action:", int(action))

State: [10.       10.       49.       69.       13.333333 92.        0.      ]
State type: <class 'numpy.ndarray'>
State dtype: float32
State shape: (7,)

DQN prediction successful!
Action: 2


In [18]:
# ============================================================
# DQN TEST EVALUATION
# ============================================================

test_env = SAEWarningEnv(test_df)

y_true = []
y_pred = []
rewards = []

state, info = test_env.reset(seed=123)

while True:

    action, _ = model.predict(
        state,
        deterministic=True
    )

    action = int(action)

    # Current state's future SAE target
    current_row = test_env.current_episode.iloc[
        test_env.current_step
    ]

    future_sae = int(
        current_row["future_sae_1h"]
    )

    prediction = int(action > 0)

    y_true.append(future_sae)
    y_pred.append(prediction)

    (
        next_state,
        reward,
        terminated,
        truncated,
        info
    ) = test_env.step(action)

    rewards.append(reward)

    if terminated or truncated:
        break

    state = next_state

In [19]:
precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print("DQN RESULTS")
print("====================")

print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)

print("\nTotal test reward:")
print(sum(rewards))

print("\nConfusion matrix:")
print(cm)

DQN RESULTS
Precision: 0.058823529411764705
Recall   : 0.3333333333333333
F1       : 0.1

Total test reward:
82.0

Confusion matrix:
[[119  16]
 [  2   1]]


In [20]:
results = pd.DataFrame({
    "Model": [
        "GCS Baseline",
        "DQN"
    ],
    "Precision": [
        precision_score(
            test_df["future_sae_1h"],
            [
                int(
                    gcs_baseline_action(row) > 0
                )
                for _, row in test_df.iterrows()
            ],
            zero_division=0
        ),
        precision_score(
            y_true,
            y_pred,
            zero_division=0
        )
    ],
    "Recall": [
        recall_score(
            test_df["future_sae_1h"],
            [
                int(
                    gcs_baseline_action(row) > 0
                )
                for _, row in test_df.iterrows()
            ],
            zero_division=0
        ),
        recall_score(
            y_true,
            y_pred,
            zero_division=0
        )
    ],
    "F1": [
        f1_score(
            test_df["future_sae_1h"],
            [
                int(
                    gcs_baseline_action(row) > 0
                )
                for _, row in test_df.iterrows()
            ],
            zero_division=0
        ),
        f1_score(
            y_true,
            y_pred,
            zero_division=0
        )
    ]
})

display(results)

,Model,Precision,Recall,F1
0,GCS Baseline,0.009383,0.538462,0.018445
1,DQN,0.058824,0.333333,0.100000


In [21]:
results_path = (
    PROJECT_ROOT /
    "data/processed/mimic3/sae_rl_results.csv"
)

results.to_csv(
    results_path,
    index=False
)

print("Results saved:")
print(results_path)

Results saved:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/data/processed/mimic3/sae_rl_results.csv
